In [23]:
import pandas as pd
from scipy.stats import mannwhitneyu
from utils import process_experiments_df
import glob

In [24]:
df_new = pd.read_csv("../results/experiment_solvers.csv")
df_new = process_experiments_df(df_new)

In [25]:
dfs_old = []
for file in glob.glob(f"../results/experiment_solvers_d*.csv"):
    dfs_old.append(pd.read_csv(file))
df_old = pd.concat(dfs_old, ignore_index=True)
df_old = process_experiments_df(df_old)

In [26]:
key_columns = ["test case", "p", "mesh family", "fine m", "solvers m", "coarse m", "solver"]

In [27]:
df_new.set_index(key_columns, inplace=True)
df_old.set_index(key_columns, inplace=True)

In [28]:
df = df_old.join(
    other=df_new,
    lsuffix="_old",
    rsuffix="_new",
)

In [33]:
results = []
for index, row in df.iterrows():
    times_old = row["solve times_old"]
    times_new = row["solve times_new"]

    if times_old is None and times_new is not None:
        print(f"New solver succeeded where old one failed: {index}")
        continue
    elif times_new is None and times_old is not None:
        print(f"Old solver succeeded where new one failed: {index}")
        continue
    elif times_old is None and times_new is None:
        continue

    stat, p_value = mannwhitneyu(times_old, times_new, alternative="less")

    result = {
        "index": index,
        "p_value": p_value,
        "old_median": pd.Series(times_old).median(),
        "new_median": pd.Series(times_new).median(),
        "old_min": pd.Series(times_old).min(),
        "new_min": pd.Series(times_new).min(),
        "min_ratio": pd.Series(times_new).min() / pd.Series(times_old).min(),
    }
    results.append(result)
results_df = pd.DataFrame(results)
results_df.set_index("index", inplace=True)
results_df

New solver succeeded where old one failed: ('continuous coefficient 3D', 4, 'uniform(3D,5)', 'S5', 'S4', 'S4', 'CG(HybridSchwarz(torch.float64, Inv(torch.float16), CUDSS), bsr_matmul=True)')
New solver succeeded where old one failed: ('continuous coefficient 3D', 4, 'uniform(3D,5)', 'S5', 'S5', 'S4', 'CG(AdditiveSchwarz(torch.float32, Inv(torch.float16), CUDSS, const_dofs_per_coarse=False), bsr_matmul=True)')
New solver succeeded where old one failed: ('continuous coefficient 3D', 4, 'uniform(3D,5)', 'S5', 'C4', 'C4', 'CG(AdditiveSchwarz(torch.float32, Inv(torch.float16), CUDSS, const_dofs_per_coarse=False), bsr_matmul=True)')
New solver succeeded where old one failed: ('continuous coefficient 3D', 4, 'uniform(3D,5)', 'S5', 'C4', 'C4', 'CG(HybridSchwarz(torch.float64, Inv(torch.float16), CUDSS), bsr_matmul=True)')
New solver succeeded where old one failed: ('continuous coefficient 3D', 4, 'uniform(3D,5)', 'S5', 'S4', 'C4', 'CG(HybridSchwarz(torch.float64, Inv(torch.float16), CUDSS), bs

/home/train_driver/Dokumenty/Studia/dd-solvers/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/train_driver/Dokumenty/Studia/dd-solvers/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/train_driver/Dokumenty/Studia/dd-solvers/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/train_driver/Dokumenty/Studia/dd-solvers/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/train_driver/Dokumenty/Studia/dd-solvers/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empt

,p_value,old_median,new_median,old_min,new_min,min_ratio
index,,,,,,
"(continuous coefficient 3D, 2, uniform(3D,6), S6, S5, S4, CG(AdditiveSchwarz(torch.float32, Inv(torch.float16), CUDSS, const_dofs_per_coarse=False), bsr_matmul=True))",0.454861,2279.108887,2278.448486,2268.890625,2277.885498,1.003964
"(continuous coefficient 3D, 2, uniform(3D,6), S6, S5, S4, CG(HybridSchwarz(torch.float64, Inv(torch.float16), CUDSS), bsr_matmul=True))",0.999933,2119.957520,2040.145569,2115.572998,2039.470581,0.964028
"(continuous coefficient 3D, 2, uniform(3D,6), S6, S6, S4, CG(AdditiveSchwarz(torch.float32, Inv(torch.float16), CUDSS, const_dofs_per_coarse=False), bsr_matmul=True))",0.001414,2435.166260,2449.643188,2432.483887,2448.116699,1.006427
"(continuous coefficient 3D, 2, uniform(3D,6), S6, S6, S4, CG(HybridSchwarz(torch.float64, Inv(torch.float16), CUDSS), bsr_matmul=True))",0.955513,2351.473999,2348.730469,2348.351074,2348.349365,0.999999
"(continuous coefficient 3D, 2, uniform(3D,6), S6, S5, S5, CG(AdditiveSchwarz(torch.float32, Inv(torch.float16), CUDSS, const_dofs_per_coarse=False), bsr_matmul=True))",0.998899,2022.061279,1998.952209,2013.006470,1997.875854,0.992484
...,...,...,...,...,...,...
"(continuous coefficient 2D, 1, uniform(2D,12), S12, S11, C11, CG(HybridSchwarz(torch.float64, Inv(torch.float16), CUDSS), bsr_matmul=True))",0.001414,1888.346741,1894.494202,1886.969360,1893.432617,1.003425
"(continuous coefficient 2D, 1, uniform(2D,12), S12, S12, C11, CG(AdditiveSchwarz(torch.float32, Inv(torch.float16), CUDSS, const_dofs_per_coarse=False), bsr_matmul=True))",0.001414,2975.959351,3012.877197,2974.862793,3009.037598,1.011488
"(continuous coefficient 2D, 1, uniform(2D,12), S12, S12, C11, CG(HybridSchwarz(torch.float64, Inv(torch.float16), CUDSS), bsr_matmul=True))",0.001414,2139.210693,2153.732788,2138.162109,2152.792969,1.006843
